# 02 — Model analysis

This notebook compares Custom CNN, EfficientNetB0, and MobileNetV2 using saved training and evaluation artifacts. Results are measured only when matching artifacts exist. It never substitutes fabricated metrics. EfficientNetB0 remains the predetermined application model.

This application is an academic/research prototype for brain MRI image classification. It is not a medical diagnostic device and should not be used to make medical decisions. Predictions should be reviewed by qualified medical professionals.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / "config" / "config.yaml").is_file() and (path / "src").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the project root or notebooks directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.utils import load_config, resolve_path
from models.registry import MODEL_NAMES, DISPLAY_NAMES
config = load_config(PROJECT_ROOT / "config" / "config.yaml")
print(f"Project: {PROJECT_ROOT.name}; task: {config['task']}")

## Experiment comparison

Every available model pair must have matching dataset fingerprints, task, class order, threshold, and test sample counts. The table always includes all three registered models. Missing experiments remain `Not available`; incompatible results raise an actionable error.

In [ ]:
from src.compare import compare_experiments
comparison = compare_experiments(config)
display(comparison)

## Measured evaluation artifacts

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

evaluation_root = resolve_path(config["evaluation_dir"], config)
records = {}
for model_name in MODEL_NAMES:
    directory = evaluation_root / model_name
    metrics_path = directory / "metrics.json"
    print(f"\n{DISPLAY_NAMES[model_name]}")
    if not metrics_path.is_file():
        print("Not available. Training required — no measured result available.")
        continue
    record = json.loads(metrics_path.read_text(encoding="utf-8"))
    records[model_name] = record
    display(pd.DataFrame.from_dict(record["per_class"], orient="index"))
    print("Dataset fingerprint:", record["dataset_fingerprint"])
    print("Test images:", record["n_samples"], "| threshold:", record["threshold"])
    for filename in ("confusion_matrix.png", "roc_curve.png", "precision_recall_curve.png"):
        path = directory / filename
        if path.is_file():
            display(Image(filename=str(path), width=650))

## Training curves and overfitting

Inspect both loss and accuracy. A growing training/validation gap can suggest overfitting. Use validation data to select training changes and leave the test population for final evaluation. A curve alone cannot establish why a model generalizes poorly.

In [ ]:
training_root = resolve_path(config["training_dir"], config)
for model_name in MODEL_NAMES:
    print(DISPLAY_NAMES[model_name])
    directory = training_root / model_name
    curve_paths = sorted(directory.glob("*.png")) if directory.is_dir() else []
    if not curve_paths:
        print("Not available. Training required — no measured training curves available.")
    for path in curve_paths:
        print(path.name)
        display(Image(filename=str(path), width=800))

## Inspect prediction errors

Per-image errors can reveal labeling or acquisition issues. After examining test errors, do not report further test-driven model revisions as an unbiased single held-out experiment. A new locked external test population is needed for independent confirmation.

In [ ]:
for model_name in records:
    path = evaluation_root / model_name / "predictions.csv"
    if path.is_file():
        predictions = pd.read_csv(path)
        errors = predictions.loc[predictions["true_label"] != predictions["predicted_label"]]
        print(f"{DISPLAY_NAMES[model_name]}: {len(errors)} misclassified test images")
        display(errors.head(20))

## Reporting

Sensitivity describes missed tumor-labelled images; specificity describes false-positive behavior. Precision depends on the evaluated class prevalence. Undefined values are unavailable, not zero or perfect. Raw output probabilities are not calibrated diagnostic confidence. Grad-CAM is an interpretability aid, not evidence of medically correct tumor localization. Include source version, manifest fingerprint, exclusions, class counts, training budget, and patient-independence limitations in the report.